In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW procedure_mapping AS
WITH concept_relationship_maps_to_procedure_dedup AS (
  SELECT
    _exponent.omop.concept_relationship.concept_id_1 AS source_concept_id,
    _exponent.omop.concept_relationship.concept_id_2 AS standard_concept_id,
    ROW_NUMBER() OVER (
      PARTITION BY _exponent.omop.concept_relationship.concept_id_1
      ORDER BY _exponent.omop.concept_relationship.concept_id_2 ASC
    ) AS row_number_within_source
  FROM _exponent.omop.concept_relationship
  INNER JOIN _exponent.omop.concept AS target_concept
    ON target_concept.concept_id =
       _exponent.omop.concept_relationship.concept_id_2
   AND target_concept.standard_concept = 'S'
   AND target_concept.invalid_reason IS NULL
   AND target_concept.domain_id = 'Procedure'
  WHERE _exponent.omop.concept_relationship.relationship_id = 'Maps to'
    AND _exponent.omop.concept_relationship.invalid_reason IS NULL
)

SELECT
  _exponent.omop.concept.concept_id AS source_concept_id,
  _exponent.omop.concept.concept_code AS source_concept_code,
  _exponent.omop.concept.vocabulary_id,
  _exponent.omop.concept.domain_id,
  _exponent.omop.concept.concept_class_id,
  concept_relationship_maps_to_procedure_dedup.standard_concept_id
FROM _exponent.omop.concept
LEFT JOIN concept_relationship_maps_to_procedure_dedup
  ON concept_relationship_maps_to_procedure_dedup.source_concept_id =
     _exponent.omop.concept.concept_id
 AND concept_relationship_maps_to_procedure_dedup.row_number_within_source = 1
WHERE _exponent.omop.concept.vocabulary_id IN ('CPT4', 'HCPCS')
  AND _exponent.omop.concept.domain_id = 'Procedure'
  AND _exponent.omop.concept.invalid_reason IS NULL;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW procedure_occurrence_charge AS
WITH primary_modifier AS (
  SELECT
    dbo_charge_modifier.chargeid AS chargeid,
    NULLIF(
      REGEXP_REPLACE(CAST(dbo_cpt4_modifier_de.entrycode AS STRING), '[\\s\\u00A0]+', ''),
      ''
    ) AS modifier_source_value,
    ROW_NUMBER() OVER (
      PARTITION BY dbo_charge_modifier.chargeid
      ORDER BY dbo_charge_modifier.modifiernumber ASC
    ) AS rn
  FROM _exponent._bronze_allscripts_tw_works.dbo_charge_modifier AS dbo_charge_modifier
  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_cpt4_modifier_de AS dbo_cpt4_modifier_de
    ON dbo_cpt4_modifier_de.id = dbo_charge_modifier.billingchargemodifierde
)
SELECT DISTINCT
  CONCAT_WS(chr(31), 'allscripts_tw', 'dbo_charge', 'id', CAST(dbo_charge.id AS BIGINT)) AS procedure_occurrence_source_value,
  source_to_person.person_id AS person_id,
  COALESCE(procedure_mapping.standard_concept_id, 0) AS procedure_concept_id,
  CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS DATE) AS procedure_date,
  CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS TIMESTAMP) AS procedure_datetime,
  CAST(44814649 AS INT) AS procedure_type_concept_id,
  CAST(0 AS INT) AS modifier_concept_id,
  CAST(COALESCE(dbo_charge.unitstobillfor, 1) AS DOUBLE) AS quantity,
  source_to_provider.provider_id AS provider_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
  NULL AS visit_detail_id,
  NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') AS procedure_source_value,
  COALESCE(procedure_mapping.source_concept_id, 0) AS procedure_source_concept_id,
  primary_modifier.modifier_source_value AS modifier_source_value,
  'allscripts_tw' AS source_system,
  CONCAT_WS(
    '|',
    CAST(COALESCE(source_to_person.person_id, 0) AS STRING),
    CAST(COALESCE(procedure_mapping.standard_concept_id, 0) AS STRING),
    CAST(CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS TIMESTAMP) AS STRING),
    CAST(COALESCE(source_to_provider.provider_id, 0) AS STRING),
    CAST(COALESCE(source_to_visit_occurrence.visit_occurrence_id, 0) AS STRING),
    COALESCE(NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), ''), '')
  ) AS dedupe_key,
  CAST(2 AS INT) AS source_priority
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge AS dbo_charge
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de AS dbo_charge_code_de
  ON dbo_charge_code_de.id = dbo_charge.chargecodede
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other AS dbo_encounter_other
  ON dbo_encounter_other.encounterid = dbo_charge.encounterid
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter AS dbo_encounter
  ON dbo_encounter.id = dbo_encounter_other.encounterid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person AS dbo_person
  ON dbo_person.id = CAST(dbo_encounter.patientid AS BIGINT)
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_provider AS dbo_provider
  ON dbo_provider.id = CAST(COALESCE(dbo_charge.billingproviderid, dbo_charge.otherproviderid) AS BIGINT)
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit AS dbo_visit
  ON dbo_visit.id = dbo_encounter.visitid
LEFT JOIN primary_modifier AS primary_modifier
  ON primary_modifier.chargeid = dbo_charge.id
 AND primary_modifier.rn = 1
JOIN procedure_mapping AS procedure_mapping
  ON procedure_mapping.source_concept_code =
     NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '')
 AND LOWER(procedure_mapping.vocabulary_id) IN ('cpt4', 'hcpcs')
JOIN _exponent.omop_mapping.source_to_person AS source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
LEFT JOIN _exponent.omop_mapping.source_to_provider AS source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(dbo_provider.id AS BIGINT)
     )
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence AS source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_visit',
       'id',
       CAST(dbo_visit.id AS BIGINT)
     )
WHERE 1=1
  AND dbo_charge.id IS NOT NULL 
  AND dbo_charge.islevelofservicechargeflag = 'N'
  AND dbo_charge_code_de.islevelofserviceflag = 'N'
  AND NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') IS NOT NULL
  AND dbo_charge.etl_load_ts BETWEEN CURRENT_TIMESTAMP() - INTERVAL 365 DAYS AND CURRENT_TIMESTAMP();

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW procedure_occurrence_order AS
WITH cte_order_activity_ranked AS (
  SELECT
    dbo_order_activity.*,
    ROW_NUMBER() OVER (
      PARTITION BY dbo_order_activity.ordernumberext
      ORDER BY
        dbo_order_activity.createddttm DESC,
        dbo_order_activity.orderactivityid DESC
    ) AS row_number
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity AS dbo_order_activity
),
cte_order_activity AS (
  SELECT
    cte_order_activity_ranked.*
  FROM cte_order_activity_ranked
  WHERE cte_order_activity_ranked.row_number = 1
)
SELECT DISTINCT
  CONCAT_WS(chr(31), 'allscripts_tw', 'dbo_item_result', 'id', CAST(dbo_item_result.id AS BIGINT)) AS procedure_occurrence_source_value,
  source_to_person.person_id AS person_id,
  COALESCE(procedure_mapping.standard_concept_id, 0) AS procedure_concept_id,
  CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS DATE) AS procedure_date,
  CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS TIMESTAMP) AS procedure_datetime,
  CAST(32817 AS INT) AS procedure_type_concept_id,
  CAST(0 AS INT) AS modifier_concept_id,
  CAST(1 AS DOUBLE) AS quantity,
  source_to_provider.provider_id AS provider_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id,
  NULL AS visit_detail_id,
  COALESCE(
    NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), ''),
    NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), '')
  ) AS procedure_source_value,
  COALESCE(procedure_mapping.source_concept_id, 0) AS procedure_source_concept_id,
  NULLIF(REGEXP_REPLACE(dbo_qo_mod_de.entrycode, '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
  'allscripts_tw' AS source_system,
  CONCAT_WS(
    '|',
    CAST(COALESCE(source_to_person.person_id, 0) AS STRING),
    CAST(COALESCE(procedure_mapping.standard_concept_id, 0) AS STRING),
    CAST(CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS TIMESTAMP) AS STRING),
    CAST(COALESCE(source_to_provider.provider_id, 0) AS STRING),
    CAST(COALESCE(source_to_visit_occurrence.visit_occurrence_id, 0) AS STRING),
    COALESCE(
      NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), ''),
      NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), '')
    )
  ) AS dedupe_key,
  CAST(1 AS INT) AS source_priority
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header AS dbo_order_activity_header
INNER JOIN cte_order_activity AS dbo_order_activity
  ON dbo_order_activity.orderactivityheaderid = dbo_order_activity_header.id
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter AS dbo_encounter
  ON dbo_encounter.id = dbo_order_activity_header.encounterid
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result AS dbo_item_result
  ON dbo_item_result.orderitemext = dbo_order_activity.ordernumberext
 AND dbo_item_result.patientid = dbo_encounter.patientid
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de AS dbo_qo_classification_de
  ON dbo_qo_classification_de.id = dbo_item_result.qoclassificationde
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_mod_de AS dbo_qo_mod_de
  ON dbo_qo_mod_de.id = COALESCE(dbo_item_result.qomod1de, dbo_item_result.qomod2de, dbo_item_result.qomod3de)
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit AS dbo_visit
  ON dbo_visit.id = dbo_encounter.visitid
JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person AS dbo_person
  ON dbo_person.id = CAST(dbo_encounter.patientid AS BIGINT)
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_provider AS dbo_provider
  ON dbo_provider.id = CAST(dbo_order_activity.orderingproviderid AS BIGINT)
JOIN procedure_mapping AS procedure_mapping
  ON procedure_mapping.source_concept_code = COALESCE(
       NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), ''),
       NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), '')
     )
 AND LOWER(procedure_mapping.vocabulary_id) IN ('cpt4', 'hcpcs')
JOIN _exponent.omop_mapping.source_to_person AS source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
LEFT JOIN _exponent.omop_mapping.source_to_provider AS source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(dbo_provider.id AS BIGINT)
     )
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence AS source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_visit',
       'id',
       CAST(dbo_visit.id AS BIGINT)
     )
WHERE 1=1
  AND dbo_item_result.id IS NOT NULL
  AND dbo_order_activity_header.activitytype = 'Order'
  AND dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
  AND dbo_qo_classification_de.itemtype = 'OT'
  AND dbo_qo_classification_de.ordertype <> 'L'
  AND COALESCE(
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code, '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode, '[\\s\\u00A0]+', ''), '')
      ) IS NOT NULL
  AND dbo_order_activity_header.etl_load_ts BETWEEN CURRENT_TIMESTAMP() - INTERVAL 365 DAYS AND CURRENT_TIMESTAMP();

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW procedure_occurrence_silver AS
WITH unioned AS (
  SELECT * FROM procedure_occurrence_order
  UNION ALL
  SELECT * FROM procedure_occurrence_charge
),
deduped AS (
  SELECT
    unioned.*,
    ROW_NUMBER() OVER (
      PARTITION BY procedure_occurrence_source_value, dedupe_key
      ORDER BY source_priority ASC, procedure_datetime ASC
    ) AS rn
  FROM unioned
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system
FROM deduped
WHERE rn = 1;


In [0]:
%sql
MERGE INTO _exponent.omop_silver.procedure_occurrence AS target
USING procedure_occurrence_silver AS source
ON target.procedure_occurrence_source_value = source.procedure_occurrence_source_value

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.procedure_concept_id <=> source.procedure_concept_id
 AND target.procedure_date <=> source.procedure_date
 AND target.procedure_datetime <=> source.procedure_datetime
 AND target.procedure_type_concept_id <=> source.procedure_type_concept_id
 AND target.modifier_concept_id <=> source.modifier_concept_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.procedure_source_value <=> source.procedure_source_value
 AND target.procedure_source_concept_id <=> source.procedure_source_concept_id
 AND target.modifier_source_value <=> source.modifier_source_value
 AND target.source_system <=> source.source_system
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.procedure_concept_id = source.procedure_concept_id,
  target.procedure_date = source.procedure_date,
  target.procedure_datetime = source.procedure_datetime,
  target.procedure_type_concept_id = source.procedure_type_concept_id,
  target.modifier_concept_id = source.modifier_concept_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.procedure_source_value = source.procedure_source_value,
  target.procedure_source_concept_id = source.procedure_source_concept_id,
  target.modifier_source_value = source.modifier_source_value,
  target.source_system = source.source_system,
  target.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
) VALUES (
  source.procedure_occurrence_source_value,
  source.person_id,
  source.procedure_concept_id,
  source.procedure_date,
  source.procedure_datetime,
  source.procedure_type_concept_id,
  source.modifier_concept_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.procedure_source_value,
  source.procedure_source_concept_id,
  source.modifier_source_value,
  source.source_system,
  CURRENT_TIMESTAMP()
);


In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
  source_system,
  procedure_occurrence_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp,
  merge_id,
  merge_reason
)
SELECT
  source_distinct.source_system,
  source_distinct.procedure_occurrence_source_value,
  TRUE AS active_flag,
  CURRENT_TIMESTAMP() AS created_tsp,
  COALESCE(source_distinct.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
  NULL AS merge_id,
  NULL AS merge_reason
FROM (
  SELECT DISTINCT
    source_system,
    procedure_occurrence_source_value,
    last_mod_tsp
  FROM _exponent.omop_silver.procedure_occurrence
  WHERE procedure_occurrence_source_value IS NOT NULL
) source_distinct
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence existing
  ON source_distinct.procedure_occurrence_source_value = existing.procedure_occurrence_source_value;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_procedure_occurrence AS
  SELECT
    source_to_procedure_occurrence.procedure_occurrence_id,
    procedure_occurrence.procedure_occurrence_source_value,
    procedure_occurrence.person_id,
    procedure_occurrence.procedure_concept_id,
    procedure_occurrence.procedure_date,
    procedure_occurrence.procedure_datetime,
    procedure_occurrence.procedure_type_concept_id,
    procedure_occurrence.modifier_concept_id,
    procedure_occurrence.quantity,
    procedure_occurrence.provider_id,
    procedure_occurrence.visit_occurrence_id,
    procedure_occurrence.visit_detail_id,
    procedure_occurrence.procedure_source_value,
    procedure_occurrence.procedure_source_concept_id,
    procedure_occurrence.modifier_source_value,
    procedure_occurrence.source_system,
    procedure_occurrence.last_mod_tsp
  FROM _exponent.omop_silver.procedure_occurrence
  JOIN _exponent.omop_mapping.source_to_procedure_occurrence
    ON procedure_occurrence.procedure_occurrence_source_value = source_to_procedure_occurrence.procedure_occurrence_source_value
  AND source_to_procedure_occurrence.active_flag = true
  WHERE procedure_occurrence.source_system = 'allscripts_tw';


In [0]:
%sql
-- MERGE INTO _exponent.omop.procedure_occurrence AS target
MERGE INTO _exponent.omop_tw.procedure_occurrence AS target
USING gold_procedure_occurrence AS source
ON target.procedure_occurrence_id = source.procedure_occurrence_id

WHEN MATCHED AND NOT (
     target.person_id <=> source.person_id
 AND target.procedure_concept_id <=> source.procedure_concept_id
 AND target.procedure_date <=> source.procedure_date
 AND target.procedure_datetime <=> source.procedure_datetime
 AND target.procedure_type_concept_id <=> source.procedure_type_concept_id
 AND target.modifier_concept_id <=> source.modifier_concept_id
 AND target.quantity <=> source.quantity
 AND target.provider_id <=> source.provider_id
 AND target.visit_occurrence_id <=> source.visit_occurrence_id
 AND target.visit_detail_id <=> source.visit_detail_id
 AND target.procedure_source_value <=> source.procedure_source_value
 AND target.procedure_source_concept_id <=> source.procedure_source_concept_id
 AND target.modifier_source_value <=> source.modifier_source_value
) THEN UPDATE SET
  target.person_id = source.person_id,
  target.procedure_concept_id = source.procedure_concept_id,
  target.procedure_date = source.procedure_date,
  target.procedure_datetime = source.procedure_datetime,
  target.procedure_type_concept_id = source.procedure_type_concept_id,
  target.modifier_concept_id = source.modifier_concept_id,
  target.quantity = source.quantity,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.procedure_source_value = source.procedure_source_value,
  target.procedure_source_concept_id = source.procedure_source_concept_id,
  target.modifier_source_value = source.modifier_source_value

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
) VALUES (
  source.procedure_occurrence_id,
  source.person_id,
  source.procedure_concept_id,
  source.procedure_date,
  source.procedure_datetime,
  source.procedure_type_concept_id,
  source.modifier_concept_id,
  source.quantity,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.procedure_source_value,
  source.procedure_source_concept_id,
  source.modifier_source_value
);
